# 03 - Plotly Interactive Plots
Plotly charts can be hovered, zoomed, and panned. Click legend items to hide/show a series.

We use `plotly.express` (`px`) because it needs very little code.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

np.random.seed(1)
dates = pd.date_range('2025-01-01', periods=365)
df = pd.DataFrame({
    'date': dates,
    'sales': np.cumsum(np.random.randn(365)) + 100,
    'region': np.random.choice(['North', 'South', 'East', 'West'], 365),
    'profit': np.random.uniform(5, 50, 365),
})
df.head()

,date,sales,region,profit
0,2025-01-01,101.624345,East,40.475147
1,2025-01-02,101.012589,West,49.134145
2,2025-01-03,100.484417,East,35.895343
3,2025-01-04,99.411449,North,46.397625
4,2025-01-05,100.276856,East,8.122818


## 1. Interactive line chart

In [2]:
fig = px.line(df, x='date', y='sales', title='Daily Sales')
fig.show()

## 2. Interactive scatter

In [3]:
fig = px.scatter(df, x='sales', y='profit', color='region',
                 hover_data=['date'], title='Sales vs Profit')
fig.show()

## 3. Bar chart

In [4]:
by_region = df.groupby('region', as_index=False)['sales'].mean()
fig = px.bar(by_region, x='region', y='sales', color='region', title='Average Sales by Region')
fig.show()

## 4. Histogram and box plot

In [5]:
px.histogram(df, x='profit', nbins=25, color='region', title='Profit Distribution').show()
px.box(df, x='region', y='profit', title='Profit by Region').show()

## 5. Pie chart

In [6]:
px.pie(by_region, names='region', values='sales', title='Sales Share').show()

## 6. Heatmap

In [7]:
df['month'] = df['date'].dt.month_name()
df['weekday'] = df['date'].dt.day_name()
pivot = df.pivot_table(index='weekday', columns='region', values='profit', aggfunc='mean')
px.imshow(pivot, text_auto='.1f', color_continuous_scale='Blues', title='Avg Profit: Weekday x Region').show()

## 7. Treemap and sunburst
Show hierarchy (region > month).

In [8]:
tm = df.groupby(['region', 'month'], as_index=False)['profit'].sum()
px.treemap(tm, path=['region', 'month'], values='profit', title='Treemap').show()
px.sunburst(tm, path=['region', 'month'], values='profit', title='Sunburst').show()

## 8. 3D scatter

In [9]:
x, y = np.random.randn(2, 200)
z = x * y + np.random.randn(200) * 0.3
fig = px.scatter_3d(x=x, y=y, z=z, color=z, title='3D Scatter')
fig.show()

## 9. 3D surface

In [10]:
xs = np.linspace(-3, 3, 50)
X, Y = np.meshgrid(xs, xs)
Z = np.sin(X ** 2 + Y ** 2) / (1 + X ** 2 + Y ** 2)
fig = go.Figure(go.Surface(x=xs, y=xs, z=Z))
fig.update_layout(title='3D Surface', scene=dict(zaxis_title='z'))
fig.show()

## 10. Animated scatter
Press **Play** or drag the slider.

In [11]:
anim = pd.DataFrame({
    'year': np.repeat([2021, 2022, 2023, 2024], 5),
    'product': ['A', 'B', 'C', 'D', 'E'] * 4,
    'price': np.random.uniform(10, 100, 20),
    'units': np.random.uniform(100, 1000, 20),
    'revenue': np.random.uniform(20, 100, 20),
})
fig = px.scatter(anim, x='price', y='units', size='revenue', color='product',
                 animation_frame='year', animation_group='product',
                 range_x=[0, 110], range_y=[0, 1100], size_max=40,
                 title='Animated Scatter')
fig.show()

## 11. Animated bar chart

In [12]:
fig = px.bar(anim, x='product', y='units', color='product',
             animation_frame='year', range_y=[0, 1100], title='Animated Bars')
fig.show()

## 12. Line chart with range slider and buttons

In [13]:
fig = px.line(df, x='date', y='sales', title='Sales with Range Slider')
fig.update_xaxes(rangeslider_visible=True,
                 rangeselector=dict(buttons=[
                     dict(count=1, label='1m', step='month', stepmode='backward'),
                     dict(count=3, label='3m', step='month', stepmode='backward'),
                     dict(step='all')]))
fig.show()

## 13. Candlestick (finance)

In [14]:
d = df.head(30)
open_ = d['sales']
close = d['sales'] + np.random.randn(30)
fig = go.Figure(go.Candlestick(x=d['date'], open=open_, close=close,
                               high=np.maximum(open_, close) + 1, low=np.minimum(open_, close) - 1))
fig.update_layout(title='Candlestick', xaxis_rangeslider_visible=False)
fig.show()

## Export to HTML
The HTML file works in any browser, no Python needed.

In [15]:
fig = px.scatter(df, x='sales', y='profit', color='region', title='Exported Chart')
fig.write_html('html_exports/scatter.html')
print('Saved: html_exports/scatter.html')

Saved: html_exports/scatter.html


In [16]:
# Save several charts into one HTML page
figs = [px.line(df, x='date', y='sales', title='Sales'),
        px.box(df, x='region', y='profit', title='Profit by Region')]
with open('html_exports/combined.html', 'w') as f:
    for i, fg in enumerate(figs):
        f.write(fg.to_html(full_html=False, include_plotlyjs='cdn' if i == 0 else False))
print('Saved: html_exports/combined.html')

Saved: html_exports/combined.html
